# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

### Ranked action queue

The playbook ranks content for human review rather than automatically changing content.

I will prioritize pages using the validated Logistic Regression ranking and keep the Week-4 action reasons as human-readable context.

**Action mapping:**

- **Quick win** — low March search impressions suggest the page is worth reviewing for an opportunity to improve visibility.
- **Refresh** — older content is worth reviewing for possible updating or improvement.
- **Quick win + Refresh** — both signals are present, so the page has two reasons for review.

The model score is a ranking signal, not a guarantee that a page will improve. The reason codes explain why a page entered the review queue.

In [4]:
# Section 1: Build ranked action queue with honest out-of-fold scores

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# -----------------------------
# Setup
# -----------------------------
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

# -----------------------------
# March 2026 features
# -----------------------------
march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(COALESCE(ga4_sessions, 0)) AS march_ga4_sessions,
        SUM(COALESCE(ga4_engaged_sessions, 0)) AS march_ga4_engaged_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
""").df()

# -----------------------------
# April 2026 observed outcome
# -----------------------------
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
""").df()

# -----------------------------
# Content metadata
# -----------------------------
content = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
""").df()

# -----------------------------
# Build modeling dataset
# -----------------------------
df = (
    march
    .merge(
        content,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        april,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

# Content age at decision cutoff
df["content_age_days"] = (
    pd.Timestamp("2026-03-31")
    - pd.to_datetime(df["content_created_date"])
).dt.days

# March CTR
df["march_ctr"] = np.where(
    df["march_impressions"] > 0,
    df["march_clicks"] / df["march_impressions"],
    0
)

# Same candidate universe used in Weeks 4–6
df = df[df["march_impressions"] > 0].copy()

# Observed future outcome
df["positive_movement"] = (
    df["april_impressions"] > df["march_impressions"]
).astype(int)

# -----------------------------
# Final Week-5 feature set
# -----------------------------
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_ga4_sessions",
    "march_ga4_engaged_sessions",
    "march_ctr",
    "content_age_days"
]

X = df[feature_cols].copy()
y = df["positive_movement"].copy()
groups = df["client_hash_id"].copy()

# -----------------------------
# 5-fold client-grouped OOF scores
# -----------------------------
def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

gkf = GroupKFold(n_splits=5)

oof_score = np.full(len(df), np.nan)

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):
    model = make_model()

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    oof_score[test_idx] = model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

df["model_score"] = oof_score

# -----------------------------
# Human-readable action reasons
# -----------------------------
def get_reason(row):
    reasons = []

    if row["march_impressions"] < 10:
        reasons.append("QUICK_WIN_VOLUME")

    if row["content_age_days"] >= 365:
        reasons.append("REFRESH_STALE")
    elif row["content_age_days"] >= 180:
        reasons.append("REFRESH_STALE")

    if not reasons:
        return "MODEL_REVIEW"

    return "+".join(reasons)

df["reason_code"] = df.apply(get_reason, axis=1)

def get_action(row):
    has_quick_win = "QUICK_WIN_VOLUME" in row["reason_code"]
    has_refresh = "REFRESH_STALE" in row["reason_code"]

    if has_quick_win and has_refresh:
        return "Quick win + Refresh"
    elif has_quick_win:
        return "Quick win"
    elif has_refresh:
        return "Refresh"
    else:
        return "Model review"

df["action"] = df.apply(get_action, axis=1)

# -----------------------------
# Rank the queue
# -----------------------------
queue = (
    df[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "action",
            "reason_code",
            "march_impressions",
            "content_age_days"
        ]
    ]
    .sort_values(
        ["model_score", "march_impressions"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

# -----------------------------
# Output checks
# -----------------------------
print("RANKED ACTION QUEUE")
print("=" * 50)

print("Queue rows:", len(queue))
print("Unique clients:", queue["client_hash_id"].nunique())
print("Missing model scores:", queue["model_score"].isna().sum())

print()
print("Top 10 recommendations:")
print(
    queue.head(10).to_string(index=False)
)

print()
print("Action counts:")
print(queue["action"].value_counts())

print()
print("Reason code counts:")
print(queue["reason_code"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

RANKED ACTION QUEUE
Queue rows: 176737
Unique clients: 47
Missing model scores: 0

Top 10 recommendations:
 rank          client_hash_id          content_hash_id  model_score       action   reason_code  march_impressions  content_age_days
    1 client_23a62021009f63c4 content_f7c9fcc26f6e23c1          1.0      Refresh REFRESH_STALE            57781.0               217
    2 client_23a62021009f63c4 content_b51957d7f4abe47e          1.0      Refresh REFRESH_STALE            85219.0               217
    3 client_20259bd6705d81d4 content_0ec90963d98b97a5          1.0 Model review  MODEL_REVIEW           130338.0               154
    4 client_73cda7b4e4f265ea content_512dbad65bd5ade9          1.0      Refresh REFRESH_STALE           154358.0               187
    5 client_23a62021009f63c4 content_36e53e9c707674fc          1.0      Refresh REFRESH_STALE           194579.0               229
    6 client_e547b89c05043229 content_eadb33b5df496f4a          1.0      Refresh REFRESH_STALE       

## 2. Intended use and limits

## Intended use and limits

This playbook is intended to help human reviewers prioritize which content to inspect first. It is decision-support, not an automatic content-changing system.

The Logistic Regression score is used to rank pages for review. The Week-4 reason codes provide human-readable context such as QUICK_WIN_VOLUME and REFRESH_STALE.

The ranking is based on observed March features and the validated model. It should be used as a prioritization signal, not as proof that a page will improve after an action.

The model should not be treated as equally reliable for every client, content type, or future period. Performance varied across held-out client groups during validation.

Human review is required before changing, removing, redirecting, or substantially rewriting content.

## 3. Human review + the no-go list

## Human review + the no-go list

A human reviewer should check the page context, search intent, current performance, and the reason code before taking action.

The reviewer should confirm that the recommended action makes sense for the specific page and that there is a real opportunity to improve it.

The following actions should not be automated:

- Publishing or substantially rewriting content.
- Deleting or redirecting a page.
- Changing important search intent or page purpose.
- Making changes based only on the model score.
- Treating the model prediction as proof that an action will improve performance.

The model is used to prioritize review. A person makes the final decision.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

### Monitoring / retrain triggers

The recommendations should be reviewed regularly because search behavior, content mix, and client patterns can change over time.

I would monitor three things:

- **Model performance:** when a new month of outcomes is available, measure Precision@20 and Precision@50 again. If performance falls to or below the Week-4 baseline, the model should be reviewed and considered for retraining.
- **Data changes:** check whether the March-style input features change substantially in later periods, especially impressions, clicks, GA4 sessions, average position, CTR, and content age.
- **Outcome/base-rate changes:** track the positive-movement rate in new evaluation periods. A large change means the old validation results may no longer represent current conditions.

Retraining should be considered when model performance drops, feature distributions change materially, or the content population changes enough that the original validation may no longer apply.

These are monitoring triggers, not automatic retraining decisions. A human should review the cause before changing the model.

In [7]:
# Section 4: Monitoring / retrain triggers

print("MONITORING / RETRAIN TRIGGERS")
print("=" * 50)

# Honest-validation reference values
baseline_p20 = 36.0
baseline_p50 = 36.4
model_p20 = 48.0
model_p50 = 46.8

print("Current honest-validation reference:")
print(f"  Model Precision@20: {model_p20:.1f}%")
print(f"  Baseline Precision@20: {baseline_p20:.1f}%")
print(f"  Model Precision@50: {model_p50:.1f}%")
print(f"  Baseline Precision@50: {baseline_p50:.1f}%")

print()
print("Retrain / review triggers:")
print("  1. New Precision@20 falls to or below the baseline.")
print("  2. New Precision@50 falls to or below the baseline.")
print("  3. Input feature distributions change materially.")
print("  4. Positive-movement base rate changes substantially.")
print("  5. Client or content mix changes enough to weaken comparability.")

print()
print("Automation rule:")
print("  No automatic content changes or automatic retraining.")
print("  Human review is required before action.")

MONITORING / RETRAIN TRIGGERS
Current honest-validation reference:
  Model Precision@20: 48.0%
  Baseline Precision@20: 36.0%
  Model Precision@50: 46.8%
  Baseline Precision@50: 36.4%

Retrain / review triggers:
  1. New Precision@20 falls to or below the baseline.
  2. New Precision@50 falls to or below the baseline.
  3. Input feature distributions change materially.
  4. Positive-movement base rate changes substantially.
  5. Client or content mix changes enough to weaken comparability.

Automation rule:
  No automatic content changes or automatic retraining.
  Human review is required before action.


## 5. Exports for the paper

### Exports for the paper

The final ranked queue is exported to `work/outputs/` so the paper can reuse the same recommendations and reason codes.

The export contains the model score, action, reason code, client/content identifiers, and the March features used for prioritization.

The queue is an observed decision-support output. It is not a production action list and should not be treated as proof that a content change will improve performance.

In [8]:
# Section 5: Export the ranked action queue and validation metrics

import os
import json

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Export ranked queue
queue_path = "work/outputs/action_playbook_queue.csv"
queue.to_csv(queue_path, index=False)

# Export validation metrics used by the playbook
metrics = {
    "model": "Logistic Regression",
    "validation": "5-fold GroupKFold by client_hash_id",
    "model_precision_at_20": round(float(model_p20), 4),
    "baseline_precision_at_20": round(float(baseline_p20), 4),
    "model_precision_at_50": round(float(model_p50), 4),
    "baseline_precision_at_50": round(float(baseline_p50), 4),
    "decision_use": "human-reviewed decision support",
    "automatic_content_changes": False,
    "automatic_retraining": False
}

metrics_path = "work/outputs/action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("EXPORTS")
print("=" * 50)
print(f"Queue exported: {queue_path}")
print(f"Queue rows: {len(queue)}")
print(f"Metrics exported: {metrics_path}")
print()
print("Exports completed successfully.")

EXPORTS
Queue exported: work/outputs/action_playbook_queue.csv
Queue rows: 176737
Metrics exported: work/outputs/action_playbook_metrics.json

Exports completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.